# Project 1 Module 3: Extract Practice

This workbook rebuilds the reasoning in `src/extract.py` with fake HTTP objects and temporary files. It never calls a real Calgary URL and never writes to `data/raw` or `outputs/logs`.

Complete each editable answer cell, then run its separate check.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime, timezone
import csv
import io


def passed(exercise: str) -> None:
    print(f"PASS: {exercise}")


PRACTICE_DATASETS = {
    "communities": {"url": "https://example.invalid/communities.geojson", "output": "data/raw/communities.geojson", "format": "geojson"},
    "roads": {"url": "https://example.invalid/roads.geojson", "output": "data/raw/roads.geojson", "format": "geojson"},
    "transit_stops": {"url": "https://example.invalid/transit_stops.geojson", "output": "data/raw/transit_stops.geojson", "format": "geojson"},
    "land_use_districts": {"url": "https://example.invalid/land_use_districts.geojson", "output": "data/raw/land_use_districts.geojson", "format": "geojson"},
}
print("Offline extract practice ready.")

Offline extract practice ready.


# Part 1: Iterate Dataset Configuration

The production config maps each dataset name to a URL, stable raw output path, and format. Iterating `.items()` preserves the name beside its configuration.

## Exercise 1A

Build `extract_plan` as `(name, output, format)` tuples in configuration order.

In [ ]:
extract_plan = []
for dataset_name, config in PRACTICE_DATASETS.items():
    # TODO: append (dataset_name, config["output"], config["format"]).
    pass
extract_plan

In [ ]:
assert [row[0] for row in extract_plan] == list(PRACTICE_DATASETS)
assert all(row[1].startswith("data/raw/") and row[2] == "geojson" for row in extract_plan)
passed("Exercise 1A")

# Part 2: Parent Directories in a Sandbox

`Path.parent.mkdir(parents=True, exist_ok=True)` creates the containing tree and tolerates an existing directory. Practice only inside `TemporaryDirectory`.

## Exercise 2A

Implement `ensure_parent_dir` and prove it does not create the file itself.

In [ ]:
def ensure_parent_dir(file_path: Path) -> None:
    # TODO: create file_path.parent recursively and idempotently.
    pass


with TemporaryDirectory() as temporary:
    nested_file = Path(temporary) / "raw" / "nested" / "sample.geojson"
    ensure_parent_dir(nested_file)
    parent_created = nested_file.parent.is_dir()
    file_created = nested_file.exists()

In [ ]:
assert parent_created is True
assert file_created is False
passed("Exercise 2A")

# Part 3: Response Contracts Without a Network

`status_code` is metadata. `raise_for_status()` turns an unsuccessful status into an exception. `content` is the response body as bytes. Extraction should validate before writing.

The completed fake below is the shared deterministic smoke fixture.

In [ ]:
class FakeHTTPError(RuntimeError):
    pass


class FakeResponse:
    def __init__(self, content: bytes, status_code: int = 200) -> None:
        self.content = content
        self.status_code = status_code

    def raise_for_status(self) -> None:
        if self.status_code >= 400:
            raise FakeHTTPError(f"HTTP {self.status_code}")


class FakeSession:
    def __init__(self, responses: dict[str, FakeResponse]) -> None:
        self.responses = responses
        self.calls: list[tuple[str, int]] = []

    def get(self, url: str, timeout: int) -> FakeResponse:
        self.calls.append((url, timeout))
        return self.responses[url]

## Exercise 3A: Validate Before Reading Bytes

Write `read_validated_response(response)` so it calls `raise_for_status()` before returning `(status_code, content)`. Use only the fake response objects above; no network request is needed.

In [ ]:
def read_validated_response(response: FakeResponse) -> tuple[int, bytes]:
    # TODO: validate first, then return status code and content.
    raise NotImplementedError("Complete Exercise 3A")

In [ ]:
class GuardedResponse:
    def __init__(self, payload: bytes, status_code: int = 200) -> None:
        self._payload = payload
        self.status_code = status_code
        self.validated = False

    def raise_for_status(self) -> None:
        self.validated = True
        if self.status_code >= 400:
            raise FakeHTTPError(f"HTTP {self.status_code}")

    @property
    def content(self) -> bytes:
        assert self.validated, "Read content only after raise_for_status()."
        return self._payload


good_3a = GuardedResponse(b'{"type": "FeatureCollection"}')
status_3a, content_3a = read_validated_response(good_3a)
assert status_3a == 200
assert content_3a.startswith(b"{")

try:
    read_validated_response(GuardedResponse(b"error", 503))
except FakeHTTPError:
    passed("Exercise 3A")
else:
    raise AssertionError("An unsuccessful response must raise before content is used.")

# Part 4: Bytes, UTC Time, and Provenance

`Path.write_bytes()` writes the exact response body and returns the number of bytes written. The production log records an aware UTC ISO timestamp plus dataset, source, destination, status, and byte count.

## Exercise 4A

Implement `persist_download`. Create only the destination parent, validate the response, write its bytes, and return one provenance row. The supplied `clock` makes the timestamp deterministic in the check.

In [ ]:
def persist_download(
    dataset_name: str,
    source_url: str,
    output_path: Path,
    response: FakeResponse,
    clock,
 ) -> dict[str, object]:
    # TODO: ensure the parent, validate, write response.content, and return the row.
    raise NotImplementedError("Complete Exercise 4A")

In [ ]:
fixed_utc_4a = datetime(2026, 1, 2, 3, 4, 5, tzinfo=timezone.utc)

with TemporaryDirectory() as temporary:
    destination_4a = Path(temporary) / "raw" / "sample.geojson"
    response_4a = FakeResponse(b"offline-bytes", 200)
    row_4a = persist_download(
        "sample",
        "https://example.invalid/sample.geojson",
        destination_4a,
        response_4a,
        lambda tz: fixed_utc_4a,
    )

    assert destination_4a.read_bytes() == b"offline-bytes"
    assert row_4a == {
        "dataset": "sample",
        "source_url": "https://example.invalid/sample.geojson",
        "output_path": str(destination_4a),
        "downloaded_at_utc": "2026-01-02T03:04:05+00:00",
        "http_status": 200,
        "bytes_written": 13,
    }

assert fixed_utc_4a.utcoffset().total_seconds() == 0
passed("Exercise 4A")

# Part 5: CSV Log Header and Append Behavior

`csv.DictWriter` needs a stable field order. In append mode, write the header only when the file does not yet exist; later calls append rows without duplicating it.

## Exercise 5A

Implement `append_provenance(log_path, rows)`. Create the parent directory, open with `newline=""` and UTF-8 encoding, write the header once, then write all rows.

In [ ]:
LOG_FIELDS = [
    "dataset",
    "source_url",
    "output_path",
    "downloaded_at_utc",
    "http_status",
    "bytes_written",
]


def append_provenance(log_path: Path, rows: list[dict[str, object]]) -> None:
    # TODO: create the parent and append rows, writing the header once.
    raise NotImplementedError("Complete Exercise 5A")

In [ ]:
row_5a = {
    "dataset": "sample",
    "source_url": "https://example.invalid/sample.geojson",
    "output_path": "/temporary/sample.geojson",
    "downloaded_at_utc": "2026-01-02T03:04:05+00:00",
    "http_status": 200,
    "bytes_written": 13,
}

with TemporaryDirectory() as temporary:
    log_5a = Path(temporary) / "logs" / "extract.csv"
    append_provenance(log_5a, [row_5a])
    append_provenance(log_5a, [{**row_5a, "dataset": "second"}])

    with log_5a.open(newline="", encoding="utf-8") as handle:
        rows_5a = list(csv.DictReader(handle))
    text_5a = log_5a.read_text(encoding="utf-8")

assert text_5a.count(",".join(LOG_FIELDS)) == 1
assert [row["dataset"] for row in rows_5a] == ["sample", "second"]
assert rows_5a[0]["bytes_written"] == "13"
passed("Exercise 5A")

# Part 6: Specific Error Handling

Broad `except Exception` blocks hide programming mistakes. Catch only failures you can label or recover from, and let unexpected errors remain visible.

## Exercise 6A

Implement `classify_download(action)`. Return `("ok", row)` on success, `("http_error", message)` for `FakeHTTPError`, and `("write_error", message)` for `OSError`. Do not catch any other exception type.

In [ ]:
def classify_download(action) -> tuple[str, object]:
    # TODO: catch FakeHTTPError and OSError separately.
    raise NotImplementedError("Complete Exercise 6A")

In [ ]:
def return_row_6a():
    return {"dataset": "sample"}


def raise_http_6a():
    raise FakeHTTPError("HTTP 429")


def raise_os_6a():
    raise OSError("disk full")


assert classify_download(return_row_6a) == ("ok", {"dataset": "sample"})
assert classify_download(raise_http_6a) == ("http_error", "HTTP 429")
assert classify_download(raise_os_6a) == ("write_error", "disk full")

try:
    classify_download(lambda: (_ for _ in ()).throw(KeyError("bug")))
except KeyError:
    passed("Exercise 6A")
else:
    raise AssertionError("Unexpected programming errors must not be swallowed.")

# Part 7: Orchestration With Injected Collaborators

Production `run_extract()` loops over `DATASETS`, downloads each configured URL, writes stable outputs, accumulates rows, then appends the log. Here the control flow is the same, but destinations are remapped beneath a caller-provided sandbox and all side effects are injected.

## Exercise 7A

Implement `orchestrate_extract`. For each dataset, build `sandbox_root / Path(config["output"]).name`, call `downloader(name, config, destination, clock)`, collect rows, call `logger(rows)` once after the loop, and return the rows.

In [ ]:
def orchestrate_extract(
    datasets: dict[str, dict[str, str]],
    sandbox_root: Path,
    downloader,
    logger,
    clock,
 ) -> list[dict[str, object]]:
    # TODO: loop, remap each output into sandbox_root, collect, log once, return.
    raise NotImplementedError("Complete Exercise 7A")

In [ ]:
calls_7a = []
logged_7a = []


def fake_downloader_7a(name, config, destination, clock):
    calls_7a.append((name, config["url"], destination))
    return {"dataset": name, "output_path": str(destination)}


def fake_logger_7a(rows):
    logged_7a.append(list(rows))


with TemporaryDirectory() as temporary:
    root_7a = Path(temporary)
    rows_7a = orchestrate_extract(
        dict(list(PRACTICE_DATASETS.items())[:2]),
        root_7a,
        fake_downloader_7a,
        fake_logger_7a,
        datetime.now,
    )
    assert all(destination.parent == root_7a for _, _, destination in calls_7a)
    assert [destination.name for _, _, destination in calls_7a] == [
        "communities.geojson", "roads.geojson"
    ]

assert [row["dataset"] for row in rows_7a] == ["communities", "roads"]
assert logged_7a == [rows_7a]
passed("Exercise 7A")

# Part 8: Debugging Drills

The function below is syntactically valid but initially has three behavioral bugs: it writes before validation, creates a naive timestamp, and reports the wrong byte count. Fix only those defects.

## Exercise 8A

Edit `debug_extract_step` so validation precedes writing, the clock receives `timezone.utc`, and `bytes_written` is based on the response bytes.

In [ ]:
def debug_extract_step(output_path: Path, response: FakeResponse, clock) -> dict[str, object]:
    # BUGS: use the check feedback to repair ordering, timezone, and byte count.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_bytes(response.content)
    response.raise_for_status()
    downloaded_at = clock()
    return {
        "downloaded_at_utc": downloaded_at.isoformat(),
        "http_status": response.status_code,
        "bytes_written": len(str(response.content)),
    }

In [ ]:
clock_calls_8a = []


def clock_8a(tz=None):
    clock_calls_8a.append(tz)
    return datetime(2026, 5, 6, 7, 8, 9, tzinfo=tz)


with TemporaryDirectory() as temporary:
    destination_8a = Path(temporary) / "payload.bin"
    row_8a = debug_extract_step(destination_8a, FakeResponse(b"abcde"), clock_8a)
    assert destination_8a.read_bytes() == b"abcde"

assert clock_calls_8a == [timezone.utc]
assert row_8a["downloaded_at_utc"].endswith("+00:00")
assert row_8a["bytes_written"] == 5

with TemporaryDirectory() as temporary:
    failed_path_8a = Path(temporary) / "must-not-exist.bin"
    try:
        debug_extract_step(failed_path_8a, FakeResponse(b"error", 500), clock_8a)
    except FakeHTTPError:
        assert not failed_path_8a.exists(), "Validation must happen before writing."
    else:
        raise AssertionError("Expected FakeHTTPError")

passed("Exercise 8A")

# Part 9: Capstone Mini Extract

Rebuild the production sequence at small scale: iterate configuration, request with a 60-second timeout, validate, write bytes, create UTC provenance rows, and append one CSV log. Every path must remain beneath `sandbox_root`, and the provided session is fake.

## Capstone

Implement `mini_extract`. Write payloads to `sandbox_root / "raw" / basename(config["output"])` and the log to `sandbox_root / "logs" / "extract_log.csv"`. Return the provenance rows. Do not import or call production `run_extract()`.

In [ ]:
def mini_extract(
    datasets: dict[str, dict[str, str]],
    sandbox_root: Path,
    session: FakeSession,
    clock,
 ) -> list[dict[str, object]]:
    # TODO: implement the complete offline extract and write one sandboxed CSV log.
    raise NotImplementedError("Complete the capstone")

In [ ]:
CAPSTONE_DATASETS = {
    "alpha": {
        "url": "https://example.invalid/alpha.geojson",
        "output": "unused/original/alpha.geojson",
        "format": "geojson",
    },
    "beta": {
        "url": "https://example.invalid/beta.geojson",
        "output": "unused/original/beta.geojson",
        "format": "geojson",
    },
}
capstone_session = FakeSession({
    "https://example.invalid/alpha.geojson": FakeResponse(b"alpha", 200),
    "https://example.invalid/beta.geojson": FakeResponse(b"beta-data", 200),
})
capstone_time = datetime(2026, 6, 7, 8, 9, 10, tzinfo=timezone.utc)

with TemporaryDirectory() as temporary:
    capstone_root = Path(temporary)
    capstone_rows = mini_extract(
        CAPSTONE_DATASETS,
        capstone_root,
        capstone_session,
        lambda tz: capstone_time,
    )

    assert (capstone_root / "raw" / "alpha.geojson").read_bytes() == b"alpha"
    assert (capstone_root / "raw" / "beta.geojson").read_bytes() == b"beta-data"
    capstone_log = capstone_root / "logs" / "extract_log.csv"
    with capstone_log.open(newline="", encoding="utf-8") as handle:
        capstone_log_rows = list(csv.DictReader(handle))
    assert all(Path(row["output_path"]).is_relative_to(capstone_root) for row in capstone_rows)

assert capstone_session.calls == [
    ("https://example.invalid/alpha.geojson", 60),
    ("https://example.invalid/beta.geojson", 60),
]
assert [row["dataset"] for row in capstone_rows] == ["alpha", "beta"]
assert [row["bytes_written"] for row in capstone_rows] == [5, 9]
assert [row["dataset"] for row in capstone_log_rows] == ["alpha", "beta"]
passed("Capstone")

# Bridge Back to `src/extract.py`

Without editing production code, describe how each practiced piece maps back to the real module:

1. Which object supplies dataset names, URLs, and output paths?
2. Why must `raise_for_status()` happen before `write_bytes()`?
3. Why is `timezone.utc` passed to `datetime.now`?
4. When should the CSV header be written?
5. What belongs in orchestration versus a single-download helper?
6. Which collaborators would you inject to unit-test orchestration offline?

Complete the plan below from memory, then compare it with `src/config.py` and `src/extract.py`.

In [ ]:
bridge_back_plan = {
    "configuration_source": "",
    "validation_before_write": "",
    "utc_reason": "",
    "header_rule": "",
    "orchestration_responsibilities": "",
    "injected_collaborators": "",
}

for topic, answer in bridge_back_plan.items():
    print(f"{topic}: {answer or '[not answered yet]'}")

# Hidden Reference Solutions

Open a section only after writing and testing your own attempt.

<details>
<summary>Exercises 1A, 2A, and 3A</summary>

```python
extract_plan = []
for dataset_name, config in PRACTICE_DATASETS.items():
    extract_plan.append((dataset_name, config["output"], config["format"]))

def ensure_parent_dir(file_path: Path) -> None:
    file_path.parent.mkdir(parents=True, exist_ok=True)

def read_validated_response(response: FakeResponse) -> tuple[int, bytes]:
    response.raise_for_status()
    return response.status_code, response.content
```
</details>

<details>
<summary>Exercise 4A</summary>

```python
def persist_download(dataset_name, source_url, output_path, response, clock):
    ensure_parent_dir(output_path)
    response.raise_for_status()
    data_bytes = response.content
    bytes_written = output_path.write_bytes(data_bytes)
    return {
        "dataset": dataset_name,
        "source_url": source_url,
        "output_path": str(output_path),
        "downloaded_at_utc": clock(timezone.utc).isoformat(),
        "http_status": response.status_code,
        "bytes_written": bytes_written,
    }
```
</details>

<details>
<summary>Exercises 5A and 6A</summary>

```python
def append_provenance(log_path, rows):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = log_path.exists()
    with log_path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=LOG_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)

def classify_download(action):
    try:
        return "ok", action()
    except FakeHTTPError as exc:
        return "http_error", str(exc)
    except OSError as exc:
        return "write_error", str(exc)
```
</details>

<details>
<summary>Exercises 7A and 8A</summary>

```python
def orchestrate_extract(datasets, sandbox_root, downloader, logger, clock):
    rows = []
    for name, config in datasets.items():
        destination = sandbox_root / Path(config["output"]).name
        rows.append(downloader(name, config, destination, clock))
    logger(rows)
    return rows

def debug_extract_step(output_path, response, clock):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    response.raise_for_status()
    data_bytes = response.content
    output_path.write_bytes(data_bytes)
    return {
        "downloaded_at_utc": clock(timezone.utc).isoformat(),
        "http_status": response.status_code,
        "bytes_written": len(data_bytes),
    }
```
</details>

<details>
<summary>Capstone</summary>

```python
def mini_extract(datasets, sandbox_root, session, clock):
    rows = []
    for name, config in datasets.items():
        destination = sandbox_root / "raw" / Path(config["output"]).name
        response = session.get(config["url"], timeout=60)
        rows.append(
            persist_download(name, config["url"], destination, response, clock)
        )
    append_provenance(sandbox_root / "logs" / "extract_log.csv", rows)
    return rows
```
</details>

# Suggested Review Schedule

- **Today:** Parts 1–4. Explain why validation precedes persistence.
- **Tomorrow:** Parts 5–7 without opening the reference solutions.
- **In three days:** Repair the debugging drill and complete the capstone.
- **In one week:** Redo the capstone with one success and one fake HTTP failure; predict what should exist afterward.
- **In two weeks:** Write the bridge-back plan from memory, then compare it with `src/config.py` and `src/extract.py`.

Focus on the sequence of decisions: iterate configuration, create parents, request, validate, capture bytes, write, timestamp in UTC, build provenance, and append the log. Exact syntax becomes easier to retrieve when the control flow is clear.